In [0]:
%sql
select * from aircat.silver.aqi_silver limit 1;

year_,month,state,area,aqi_value,number_of_monitoring_stations,processed_timestamp,aid
2015,August,Delhi,Delhi,147.1,4,2026-03-17T03:32:29.746Z,1


In [0]:
%sql
select * from aircat.silver.population_silver limit 1;

year,month,state,gender,value,processed_timestamp,pid
2026,October,West Bengal,Male,19841,2026-03-17T03:38:52.099Z,1


In [0]:
%sql
select * from aircat.silver.vehicle_silver limit 1;

year,month,state,vehicle_class,fuel,value,processed_timestamp,vid
2025,April,Andaman and Nicobar Islands,BUS,DIESEL,2,2026-03-17T03:44:03.362Z,1


In [0]:
%sql
select * from aircat.silver.health_silver limit 1;

year,reporting_date,state,disease_illness_name,cases,deaths,reporting_month,processed_timestamp,hid
2025,2025-04-05,Assam,Food Poisoning,18,0,April,2026-03-17T04:01:35.106Z,1


creating region_lookup dimension

In [0]:
%sql
create table if not exists aircat.gold.region_lookup
USING DELTA AS
select 
row_number() over(order by state) as region_lookup_id,
state, area,
dense_rank() over(order by state) as lookup_state_id,
dense_rank() over(order by area) as lookup_city_id
from (
    select distinct h.state, a.area
    from aircat.silver.health_silver  h
    join aircat.silver.aqi_silver a
    on a.state=h.state
    where h.state is not null
)



num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from aircat.gold.region_lookup

region_lookup_id,state,area,lookup_state_id,lookup_city_id
1,Andaman and Nicobar Islands,Sri Vijaya Puram,1,262
2,Andhra Pradesh,Amaravati,2,9
3,Andhra Pradesh,Anantapur,2,13
4,Andhra Pradesh,Chittoor,2,72
5,Andhra Pradesh,Kadapa,2,137
6,Andhra Pradesh,Rajamahendravaram,2,230
7,Andhra Pradesh,Tirumala,2,274
8,Andhra Pradesh,Tirupati,2,276
9,Andhra Pradesh,Vijayawada,2,290
10,Andhra Pradesh,Visakhapatnam,2,293


creating disease_lookup dimension

In [0]:
%sql
create table if not exists aircat.gold.disease_lookup
using delta as
select distinct(disease_illness_name),
dense_rank() over(order by disease_illness_name) as lookup_disease_id
from aircat.silver.health_silver
where disease_illness_name is not null;

num_affected_rows,num_inserted_rows


Creating vehicle_type_lookup dimension

In [0]:
%sql
create table if not exists aircat.gold.vehicle_type_lookup
using delta as
select
row_number() over(order by vehicle_class) as vehicle_type_lookup_id,
vehicle_class, fuel,
dense_rank() over(order by vehicle_class) as lookup_vehicle_class_id,
dense_rank() over(order by fuel) as lookup_fuel_id
from(
    select distinct vehicle_class, fuel
    from aircat.silver.vehicle_silver
)

num_affected_rows,num_inserted_rows


creating month_year_lookup dimnesion

In [0]:
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType)

schema = StructType([
  StructField('month' , StringType(), True)
])

month_data = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

month_df = spark.createDataFrame(schema=schema, data=month_data)
month_df.createOrReplaceTempView("month_temp")

In [0]:
%sql
create table if not exists aircat.gold.calender_lookup
using delta as
select
row_number() over(order by year, month) as calender_lookup_id,
year, month,
dense_rank() over(order by year) as lookup_year_id,
dense_rank() over(order by month) as lookup_month_id
from
(
select distinct h.year as year, m.month as month
          from aircat.silver.health_silver as h
          cross join month_temp as m
)

num_affected_rows,num_inserted_rows


creating state dimension

In [0]:
%sql
create table if not exists aircat.gold.state_lookup
using delta as
select 
state,
row_number() over(order by state) as state_lookup_id
from(
select distinct state
from aircat.silver.health_silver
where state is not null
)


num_affected_rows,num_inserted_rows


# Fact Tables

creating AQI Fact Table

In [0]:
%sql
create table if not exists aircat.gold.aqi_fact
USING DELTA AS
select
    a.aid as aqi_id,
    cl.calender_lookup_id,
    rl.region_lookup_id,
    a.aqi_value,
    a.number_of_monitoring_stations
from aircat.silver.aqi_silver a
left join aircat.gold.calender_lookup cl 
on a.year_=cl.year and a.month=cl.month
left join aircat.gold.region_lookup rl
on a.state=rl.state and a.area=rl.area

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from aircat.gold.aqi_fact

aqi_id,calender_lookup_id,region_lookup_id,aqi_value,number_of_monitoring_stations
1,74,52,147.1,4
2,74,53,71.26315789473684,1
3,74,66,100.23333333333333,1
4,74,93,61.758620689655174,2
5,74,149,85.0909090909091,1
6,74,159,67.2,1
7,74,163,61.758620689655174,1
8,74,166,87.9090909090909,1
9,74,239,100.16666666666667,2
10,74,264,63.8,1


creating population fact

In [0]:
%sql
create table if not exists aircat.gold.population_fact
using delta as
select
  p.pid as population_id,
  cl.calender_lookup_id,
  sl.state_lookup_id,
  p.gender,
  p.value as population_count
from aircat.silver.population_silver p 
left join aircat.gold.calender_lookup cl 
on p.year=cl.year and p.month=cl.month   
join aircat.gold.state_lookup sl 
on p.state=sl.state
where cl.calender_lookup_id < 2026
 


num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from aircat.gold.population_fact

population_id,calender_lookup_id,state_lookup_id,gender,population_count
229,203,40,Male,19545
230,203,40,Female,18450
231,203,39,Male,2352
232,203,39,Female,2079
233,203,38,Male,31155
234,203,38,Female,27852
235,203,37,Male,906
236,203,37,Female,881
237,203,36,Male,9601
238,203,36,Female,9476


creating vehicle fact

In [0]:
%sql
create table if not exists aircat.gold.vehicle_fact
using delta as
select
  a.vid as vehicle_id,
  cl.calender_lookup_id,
  sl.state_lookup_id,
  vl.vehicle_type_lookup_id,
  a.value as vehicle_count

from aircat.silver.vehicle_silver a
left join aircat.gold.calender_lookup cl 
on a.year=cl.year and a.month=cl.month
left join aircat.gold.state_lookup sl
on a.state=sl.state
left join aircat.gold.vehicle_type_lookup vl
on a.vehicle_class=vl.vehicle_class and a.fuel=vl.fuel
where cl.calender_lookup_id < 2026



num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from aircat.gold.vehicle_fact


vehicle_id,calender_lookup_id,state_lookup_id,vehicle_type_lookup_id,vehicle_count
1,193,1,85,2
2,193,1,257,23
3,193,1,268,1
4,193,1,385,1
5,193,1,391,387
6,193,1,393,83
7,193,1,396,2
8,193,1,437,1
9,193,1,489,3
10,193,1,498,1


creating health fact table

In [0]:
%sql
create table if not exists aircat.gold.health_fact
USING DELTA AS
select 
 h.hid as health_id,
 cl.calender_lookup_id,
 sl.state_lookup_id,
 h.reporting_date,
 dl.lookup_disease_id,
 h.cases,
 h.deaths
from aircat.silver.health_silver h
left join aircat.gold.calender_lookup cl
on h.year=cl.year and h.reporting_month=cl.month
left join aircat.gold.state_lookup sl
on h.state=sl.state
join aircat.gold.disease_lookup dl
on h.disease_illness_name=dl.disease_illness_name
where cl.calender_lookup_id < 2026
    

num_affected_rows,num_inserted_rows


Creating Merge/Upsert for Gold Layer Tables

merge into aqi_fact

In [0]:
%sql
MERGE INTO aircat.gold.aqi_fact AS target
USING (
      select
      a.aid as aqi_id,
      cl.calender_lookup_id,
      rl.region_lookup_id,
      a.aqi_value,
      a.number_of_monitoring_stations
  from aircat.silver.aqi_silver a
  left join aircat.gold.calender_lookup cl 
  on a.year_=cl.year and a.month=cl.month
  left join aircat.gold.region_lookup rl
  on a.state=rl.state and a.area=rl.area
) AS source

ON target.calender_lookup_id  = source.calender_lookup_id
AND target.region_lookup_id = source.region_lookup_id

WHEN MATCHED 
AND (
    target.aqi_value != source.aqi_value
    OR target.number_of_monitoring_stations != source.number_of_monitoring_stations
)
THEN UPDATE SET
    target.aqi_value = source.aqi_value,
    target.number_of_monitoring_stations = source.number_of_monitoring_stations
    
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


merge into population_fact

In [0]:
%sql
MERGE INTO aircat.gold.population_fact AS target
USING (
    SELECT 
      p.pid as population_id,
      cl.calender_lookup_id,
      sl.state_lookup_id,
      p.gender,
      p.value as population_count
    from aircat.silver.population_silver p 
    left join aircat.gold.calender_lookup cl 
    on p.year=cl.year and p.month=cl.month   
    join aircat.gold.state_lookup sl 
    on p.state=sl.state
    where cl.calender_lookup_id < 2026
) AS source

ON target.state_lookup_id = source.state_lookup_id
AND target.calender_lookup_id = source.calender_lookup_id
AND target.gender = source.gender

WHEN MATCHED 
AND (
    target.population_count != source.population_count
)
THEN UPDATE SET
    target.population_count = source.population_count

WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


merge into health_fact

In [0]:
%sql
MERGE INTO aircat.gold.health_fact AS target

USING 
(
    select 
    h.hid as health_id,
    cl.calender_lookup_id,
    sl.state_lookup_id,
    h.reporting_date,
    dl.lookup_disease_id,
    h.cases,
    h.deaths
    from aircat.silver.health_silver h
    left join aircat.gold.calender_lookup cl
    on h.year=cl.year and h.reporting_month=cl.month
    left join aircat.gold.state_lookup sl
    on h.state=sl.state
    join aircat.gold.disease_lookup dl
    on h.disease_illness_name=dl.disease_illness_name
    where cl.calender_lookup_id < 2026
) AS source

ON target.state_lookup_id = source.state_lookup_id
AND target.calender_lookup_id = source.calender_lookup_id
AND target.reporting_date = source.reporting_date
AND target.lookup_disease_id = source.lookup_disease_id

WHEN MATCHED AND (
    target.cases != source.cases
    OR target.deaths != source.deaths
)
THEN UPDATE SET 
    target.cases = source.cases,
    target.deaths = source.deaths

WHEN NOT MATCHED THEN 
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


merge into vehicle_fact

In [0]:
%sql
MERGE INTO aircat.gold.vehicle_fact AS target
USING (
        select
      a.vid as vehicle_id,
      cl.calender_lookup_id,
      sl.state_lookup_id,
      vl.vehicle_type_lookup_id,
      a.value as vehicle_count

    from aircat.silver.vehicle_silver a
    left join aircat.gold.calender_lookup cl 
    on a.year=cl.year and a.month=cl.month
    left join aircat.gold.state_lookup sl
    on a.state=sl.state
    left join aircat.gold.vehicle_type_lookup vl
    on a.vehicle_class=vl.vehicle_class and a.fuel=vl.fuel
    where cl.calender_lookup_id < 2026
) AS source

ON target.state_lookup_id = source.state_lookup_id
AND target.calender_lookup_id = source.calender_lookup_id
AND target.vehicle_type_lookup_id =source.vehicle_type_lookup_id

WHEN MATCHED
AND target.vehicle_count <> source.vehicle_count
THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
df = spark.table("aircat.gold.region_lookup").toPandas()

df.to_csv("region_lookup.csv", index=False)

_Loading gold tables into **ADLS**_

In [0]:

spark.conf.set(
  "fs.azure.account.key.aqiprojectstorageaccount.dfs.core.windows.net",
  "config key hidden (project demonstration)"
)

In [0]:
spark.table("aircat.gold.aqi_fact") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("abfss://goldtables@aqiprojectstorageaccount.dfs.core.windows.net/gold/aqi_fact/")